In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("../data/OmniRetail_Customer_Segments.xlsx")

print("Dataset shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum().sum())
print("\nSegment counts:\n", df["Customer_Segment"].value_counts().sort_index())
print("\nUnique customers:", df["Customer_ID"].nunique())

Dataset shape: (5200, 25)

Missing values:
 77

Segment counts:
 Customer_Segment
1    3028
2    1076
3    1096
Name: count, dtype: int64

Unique customers: 5200


In [2]:
profile_vars = [
    "Age",
    "Annual_Income_000",
    "Annual_Spend_000",
    "Avg_Basket_Value",
    "Loyalty_Points",
    "Satisfaction_Score",
    "Online_Purchase_Pct",
    "Promo_Purchase_Pct",
    "Online_Orders_Year",
    "Store_Visits_Year",
    "Complaints_Year",
    "Tenure_Months"
]

# Check that all requested columns exist
missing_cols = [col for col in profile_vars if col not in df.columns]
print("Columns not found:", missing_cols)

# Compare segment averages
segment_profile = (
    df.groupby("Customer_Segment")[profile_vars]
    .mean()
    .round(2)
    .T
)

display(segment_profile)

Columns not found: []


Customer_Segment,1,2,3
Age,32.14,43.61,54.51
Annual_Income_000,748.56,1661.14,1089.60
Annual_Spend_000,214.92,523.65,329.47
Avg_Basket_Value,2353.96,6155.74,3626.23
Loyalty_Points,3235.02,6772.13,6018.65
Satisfaction_Score,7.00,8.34,7.98
Online_Purchase_Pct,57.82,47.59,19.59
Promo_Purchase_Pct,50.37,16.64,20.12
Online_Orders_Year,19.29,21.80,7.48
Store_Visits_Year,11.79,23.75,35.02


In [3]:
categorical_vars = [
    "Membership_Tier",
    "Preferred_Category",
    "City_Tier",
    "Region",
    "Acquisition_Channel"
]

for col in categorical_vars:
    print(f"\n{'=' * 55}")
    print(f"{col} — Percentage within each segment")
    print(f"{'=' * 55}")

    profile = pd.crosstab(
        df["Customer_Segment"],
        df[col],
        normalize="index"
    ) * 100

    display(profile.round(1))


Membership_Tier — Percentage within each segment


Membership_Tier,Basic,Gold,Platinum,Silver,Unknown
Customer_Segment,,,,,
1,66.6,2.2,0.2,29.9,1.2
2,1.0,48.0,31.5,18.5,0.9
3,11.4,27.9,0.5,59.5,0.7



Preferred_Category — Percentage within each segment


Preferred_Category,Apparel,Beauty,Electronics,Grocery,Home,Sports,Unknown
Customer_Segment,,,,,,,
1,15.4,17.4,17.4,16.7,16.3,15.9,0.8
2,15.9,16.1,16.5,16.4,17.3,17.1,0.7
3,17.2,16.6,19.0,15.6,15.1,16.0,0.6



City_Tier — Percentage within each segment


City_Tier,Tier 1,Tier 2,Tier 3
Customer_Segment,,,
1,47.7,36.6,15.7
2,45.8,35.5,18.7
3,45.3,39.9,14.8



Region — Percentage within each segment


Region,East,North,South,Unknown,West
Customer_Segment,,,,,
1,20.3,27.8,26.8,0.8,24.3
2,20.0,29.1,27.4,0.8,22.7
3,21.0,28.1,25.5,1.1,24.3



Acquisition_Channel — Percentage within each segment


Acquisition_Channel,Marketplace,Organic,Referral,Search Ads,Social Media,Store Walk-in
Customer_Segment,,,,,,
1,12.5,19.2,13.6,17.9,17.3,19.4
2,12.4,19.8,12.5,17.8,18.3,19.2
3,11.4,19.2,15.1,18.5,16.7,19.2


In [4]:
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd

categorical_vars = [
    "Membership_Tier",
    "Preferred_Category",
    "City_Tier",
    "Region",
    "Acquisition_Channel"
]

results = []

for col in categorical_vars:

    contingency_table = pd.crosstab(
        df["Customer_Segment"],
        df[col]
    )

    chi2, p_value, dof, expected = chi2_contingency(
        contingency_table
    )

    n = contingency_table.to_numpy().sum()
    rows, cols = contingency_table.shape

    cramers_v = np.sqrt(
        chi2 / (n * min(rows - 1, cols - 1))
    )

    results.append({
        "Variable": col,
        "Chi_Square": chi2,
        "Degrees_of_Freedom": dof,
        "P_Value": p_value,
        "Cramers_V": cramers_v,
        "Minimum_Expected_Count": expected.min()
    })

association_results = pd.DataFrame(results)

# Adjust for testing multiple categorical variables
association_results["Holm_Adjusted_P"] = multipletests(
    association_results["P_Value"],
    method="holm"
)[1]

association_results = association_results.sort_values(
    "Cramers_V",
    ascending=False
).reset_index(drop=True)

display(association_results.round(4))

,Variable,Chi_Square,Degrees_of_Freedom,P_Value,Cramers_V,Minimum_Expected_Count,Holm_Adjusted_P
0,Membership_Tier,3746.4177,8,0.0000,0.6002,11.1738,0.0000
1,City_Tier,10.3388,4,0.0351,0.0315,173.6085,0.1404
2,Preferred_Category,7.7621,12,0.8034,0.0273,8.2769,1.0000
3,Acquisition_Channel,4.6275,10,0.9146,0.0211,132.0169,1.0000
4,Region,3.3247,8,0.9124,0.0179,9.3115,1.0000


In [5]:
display(
    association_results[
        [
            "Variable",
            "Minimum_Expected_Count",
            "P_Value",
            "Holm_Adjusted_P"
        ]
    ].round(4)
)

,Variable,Minimum_Expected_Count,P_Value,Holm_Adjusted_P
0,Membership_Tier,11.1738,0.0000,0.0000
1,City_Tier,173.6085,0.0351,0.1404
2,Preferred_Category,8.2769,0.8034,1.0000
3,Acquisition_Channel,132.0169,0.9146,1.0000
4,Region,9.3115,0.9124,1.0000


### Categorical Association Analysis

Chi-square tests of independence were conducted to examine the association between customer segment and five categorical variables. Holm correction was applied to account for multiple comparisons.

Membership tier showed a strong association with customer segment (χ² = 3746.42, df = 8, Cramér’s V = 0.6002, p < 0.001). Segment 1 was predominantly Basic and Silver, Segment 2 had a high concentration of Gold and Platinum members, and Segment 3 was primarily Silver and Gold.

City tier, preferred product category, acquisition channel, and region did not show statistically significant associations after Holm correction. Their Cramér’s V values were all below 0.04, indicating negligible associations in this dataset.

These findings suggest that membership tier is useful for differentiating the observed customer segments, whereas the other categorical variables offer limited additional differentiation. The results describe associations and do not establish causation.

In [6]:
strategy_matrix = pd.DataFrame([
    {
        "Segment": "1 - Digital, Promotion-Dependent",
        "Customer_Count": 3028,
        "Share_Pct": 58.23,
        "Primary_Objective": "Increase profitable repeat purchasing",
        "Recommended_Actions": (
            "Test targeted, purchase-linked offers instead of blanket discounts; "
            "investigate complaint drivers; introduce relevant loyalty milestones; "
            "encourage complementary purchases"
        ),
        "Primary_KPIs": (
            "Repeat purchase rate; promotion redemption rate; "
            "spend per customer; average basket value; complaints per customer"
        ),
        "Key_Risk": (
            "Discounts may increase sales without improving profitability; "
            "reducing incentives too quickly may discourage purchases"
        )
    },
    {
        "Segment": "2 - High-Value, Omnichannel",
        "Customer_Count": 1076,
        "Share_Pct": 20.69,
        "Primary_Objective": "Retain high-value customers and protect margins",
        "Recommended_Actions": (
            "Offer personalized loyalty benefits; test premium service experiences; "
            "recommend complementary products; provide a consistent online and "
            "in-store experience"
        ),
        "Primary_KPIs": (
            "Retention rate; spend per customer; customer lifetime value; "
            "repeat purchase rate; gross margin per customer"
        ),
        "Key_Risk": (
            "Over-discounting may erode margins; premium benefits may add cost "
            "without improving retention"
        )
    },
    {
        "Segment": "3 - Established, Store-Oriented",
        "Customer_Count": 1096,
        "Share_Pct": 21.08,
        "Primary_Objective": "Strengthen store engagement and customer value",
        "Recommended_Actions": (
            "Test in-store loyalty benefits; provide relevant complementary-product "
            "offers; improve convenience of digital ordering and account access; "
            "use purchase history for personalization"
        ),
        "Primary_KPIs": (
            "Store repeat visit rate; spend per customer; average basket value; "
            "retention rate; digital adoption rate"
        ),
        "Key_Risk": (
            "Digital initiatives may have low adoption; frequent visits alone "
            "do not guarantee higher spending or profitability"
        )
    }
])

display(strategy_matrix)

,Segment,Customer_Count,Share_Pct,Primary_Objective,Recommended_Actions,Primary_KPIs,Key_Risk
0,"1 - Digital, Promotion-Dependent",3028,58.23,Increase profitable repeat purchasing,"Test targeted, purchase-linked offers instead ...",Repeat purchase rate; promotion redemption rat...,Discounts may increase sales without improving...
1,"2 - High-Value, Omnichannel",1076,20.69,Retain high-value customers and protect margins,Offer personalized loyalty benefits; test prem...,Retention rate; spend per customer; customer l...,Over-discounting may erode margins; premium be...
2,"3 - Established, Store-Oriented",1096,21.08,Strengthen store engagement and customer value,Test in-store loyalty benefits; provide releva...,Store repeat visit rate; spend per customer; a...,Digital initiatives may have low adoption; fre...


## Executive Summary

### Objective

The objective of this analysis was to identify actionable customer segments in the OmniRetail dataset and translate customer behavior into targeted business strategies. The analysis covered 5,200 unique customers and used PCA-based K-Means clustering to identify three distinct segments.

### Key Findings

**Segment 1: Digital, Promotion-Dependent Customers — 58.23%**

This is the largest segment, comprising 3,028 customers. It has the highest online purchase share (57.82%) and promotion-purchase share (50.37%), but the lowest average annual spending (214.92), loyalty points (3,235.02), and satisfaction (7.00/10). It also records the highest average complaints (2.31). The primary opportunity is to improve repeat purchasing and customer experience while testing whether targeted incentives can reduce unnecessary discount dependence.

**Segment 2: High-Value, Omnichannel Customers — 20.69%**

This segment contains 1,076 customers and has the highest average annual spending (523.65), average basket value (6,155.74), loyalty points (6,772.13), and satisfaction (8.34/10). Gold and Platinum members together account for 79.5% of the segment. The primary opportunity is to retain high-value customers through personalized loyalty benefits, strong service, and relevant cross-selling while protecting contribution margins.

**Segment 3: Established, Store-Oriented Customers — 21.08%**

This segment comprises 1,096 customers and has the longest average tenure (111.55 months) and the highest store-visit frequency (35.02 visits per year). Its online purchase share is the lowest at 19.59%, while average spending (329.47) and satisfaction (7.98/10) are between those of the other segments. The primary opportunity is to strengthen store engagement, encourage relevant complementary purchases, and offer convenient digital options without assuming that customers want to move away from physical stores.

### Categorical Analysis

Membership tier showed a strong association with customer segment (Cramér’s V = 0.6002, Holm-adjusted p < 0.001). City tier, preferred product category, acquisition channel, and region showed negligible associations and were not statistically significant after Holm correction. Membership tier therefore offers useful context for loyalty planning, whereas the other categorical variables provide limited differentiation in this analysis.

### Recommended Business Priorities

1. **Improve profitable repeat purchasing:** Test targeted promotions and complaint-reduction initiatives for Segment 1.
2. **Protect high-value customers:** Prioritize retention, personalized benefits, and margin-aware cross-selling for Segment 2.
3. **Strengthen store-oriented engagement:** Test in-store loyalty initiatives and optional digital convenience features for Segment 3.
4. **Measure incremental impact:** Evaluate campaigns through controlled pilots and monitor retention, repeat purchase rate, spending, basket value, complaint rate, and contribution margin.

### Conclusion

The analysis identifies three customer groups with distinct purchasing patterns and commercial opportunities. PCA-based clustering produced stronger internal clustering metrics than K-Means on the original standardized features, while the Adjusted Rand Index of 0.9994 showed near-identical customer assignments between the two approaches. This supports the stability of the segmentation across feature representations.

The recommendations should be treated as hypotheses for business testing rather than proven causal effects. Future evaluation using transaction-level, campaign, profitability, and longitudinal retention data will help determine which interventions create measurable and sustainable business value.

In [7]:
kpi_tracking_plan = pd.DataFrame([
    {
        "KPI": "Average annual spending",
        "Measurement": "Mean Annual_Spend_000 per customer",
        "Current_Status": "Baseline available",
        "Business_Use": "Monitor spending and customer value",
        "Priority_Segment": "All segments"
    },
    {
        "KPI": "Average basket value",
        "Measurement": "Mean Avg_Basket_Value per customer",
        "Current_Status": "Baseline available",
        "Business_Use": "Monitor changes in basket value",
        "Priority_Segment": "Segments 1 and 3"
    },
    {
        "KPI": "Customer satisfaction",
        "Measurement": "Mean Satisfaction_Score",
        "Current_Status": "Baseline available",
        "Business_Use": "Monitor experience and satisfaction",
        "Priority_Segment": "All segments"
    },
    {
        "KPI": "Complaint burden",
        "Measurement": "Mean Complaints_Year per customer",
        "Current_Status": "Baseline available",
        "Business_Use": "Identify service problems and track improvements",
        "Priority_Segment": "Segment 1"
    },
    {
        "KPI": "Online purchase share",
        "Measurement": "Mean Online_Purchase_Pct",
        "Current_Status": "Baseline available",
        "Business_Use": "Monitor channel behavior",
        "Priority_Segment": "Segments 1 and 3"
    },
    {
        "KPI": "Promotion purchase share",
        "Measurement": "Mean Promo_Purchase_Pct",
        "Current_Status": "Baseline available",
        "Business_Use": "Monitor promotion dependence",
        "Priority_Segment": "Segment 1"
    },
    {
        "KPI": "Repeat purchase rate",
        "Measurement": "Customers purchasing again / eligible customers",
        "Current_Status": "Additional transaction history required",
        "Business_Use": "Measure repeat purchasing",
        "Priority_Segment": "Segments 1 and 2"
    },
    {
        "KPI": "Customer retention rate",
        "Measurement": "Retained customers / eligible customers",
        "Current_Status": "Multiple time periods required",
        "Business_Use": "Evaluate customer retention",
        "Priority_Segment": "Segment 2"
    },
    {
        "KPI": "Promotion redemption rate",
        "Measurement": "Customers redeeming offers / customers receiving offers",
        "Current_Status": "Campaign exposure and redemption data required",
        "Business_Use": "Evaluate offer effectiveness",
        "Priority_Segment": "Segment 1"
    },
    {
        "KPI": "Contribution margin per customer",
        "Measurement": "Customer revenue minus attributable costs",
        "Current_Status": "Revenue and cost data required",
        "Business_Use": "Measure profitability after incentives",
        "Priority_Segment": "All segments"
    },
    {
        "KPI": "Incremental campaign impact",
        "Measurement": "Outcome difference between treatment and control groups",
        "Current_Status": "Campaign experiment data required",
        "Business_Use": "Test whether interventions create incremental value",
        "Priority_Segment": "All segments"
    }
])

display(kpi_tracking_plan)

,KPI,Measurement,Current_Status,Business_Use,Priority_Segment
0,Average annual spending,Mean Annual_Spend_000 per customer,Baseline available,Monitor spending and customer value,All segments
1,Average basket value,Mean Avg_Basket_Value per customer,Baseline available,Monitor changes in basket value,Segments 1 and 3
2,Customer satisfaction,Mean Satisfaction_Score,Baseline available,Monitor experience and satisfaction,All segments
3,Complaint burden,Mean Complaints_Year per customer,Baseline available,Identify service problems and track improvements,Segment 1
4,Online purchase share,Mean Online_Purchase_Pct,Baseline available,Monitor channel behavior,Segments 1 and 3
5,Promotion purchase share,Mean Promo_Purchase_Pct,Baseline available,Monitor promotion dependence,Segment 1
6,Repeat purchase rate,Customers purchasing again / eligible customers,Additional transaction history required,Measure repeat purchasing,Segments 1 and 2
7,Customer retention rate,Retained customers / eligible customers,Multiple time periods required,Evaluate customer retention,Segment 2
8,Promotion redemption rate,Customers redeeming offers / customers receivi...,Campaign exposure and redemption data required,Evaluate offer effectiveness,Segment 1
9,Contribution margin per customer,Customer revenue minus attributable costs,Revenue and cost data required,Measure profitability after incentives,All segments


In [8]:
import pandas as pd
from IPython.display import display

# 1. Build the current profile for each segment
summary = (
    df.groupby("Customer_Segment")
    .agg(
        Customers=("Customer_ID", "nunique"),
        Annual_Spend=("Annual_Spend_000", "mean"),
        Basket_Value=("Avg_Basket_Value", "mean"),
        Satisfaction=("Satisfaction_Score", "mean"),
        Complaints=("Complaints_Year", "mean"),
        Online_Share=("Online_Purchase_Pct", "mean"),
        Promo_Share=("Promo_Purchase_Pct", "mean"),
        Store_Visits=("Store_Visits_Year", "mean"),
        Tenure=("Tenure_Months", "mean")
    )
)

summary["Customer_Share"] = (
    summary["Customers"] / df["Customer_ID"].nunique() * 100
)

# 2. Calculate membership composition within each segment
membership_mix = pd.crosstab(
    df["Customer_Segment"],
    df["Membership_Tier"],
    normalize="index"
) * 100


def membership_share(segment, tiers):
    """Return combined percentage for selected membership tiers."""
    return sum(
        membership_mix.loc[segment, tier]
        for tier in tiers
        if tier in membership_mix.columns
    )


# 3. Combine evidence, decisions, actions, KPIs and risks
decision_data = [
    {
        "Segment": "1 — Digital, Promotion-Dependent",
        "Evidence": (
            f"{summary.loc[1, 'Customers']:,} customers "
            f"({summary.loc[1, 'Customer_Share']:.1f}%). "
            f"Mean annual spend: {summary.loc[1, 'Annual_Spend']:.2f}; "
            f"satisfaction: {summary.loc[1, 'Satisfaction']:.2f}/10; "
            f"complaints: {summary.loc[1, 'Complaints']:.2f}; "
            f"online purchase share: {summary.loc[1, 'Online_Share']:.1f}%; "
            f"promotion purchase share: {summary.loc[1, 'Promo_Share']:.1f}%. "
            f"{membership_share(1, ['Basic', 'Silver']):.1f}% are Basic or Silver members."
        ),
        "Business_Decision": (
            "Improve profitable repeat purchasing and address customer "
            "experience problems without relying on blanket discounts."
        ),
        "Priority_Actions": (
            "Test targeted offers against a control group; investigate "
            "complaint drivers; introduce relevant loyalty milestones "
            "and complementary-product recommendations."
        ),
        "KPI_Baseline": (
            f"Spend: {summary.loc[1, 'Annual_Spend']:.2f}; "
            f"basket value: {summary.loc[1, 'Basket_Value']:.2f}; "
            f"satisfaction: {summary.loc[1, 'Satisfaction']:.2f}/10; "
            f"complaints: {summary.loc[1, 'Complaints']:.2f}."
        ),
        "Success_Measures": (
            "Repeat purchase rate, incremental contribution margin, "
            "promotion redemption, average basket value and complaint rate."
        ),
        "Risk_and_Data_Need": (
            "Discounts may reduce margins without creating incremental sales. "
            "Collect offer exposure, redemption, transaction and cost data."
        )
    },
    {
        "Segment": "2 — High-Value, Omnichannel",
        "Evidence": (
            f"{summary.loc[2, 'Customers']:,} customers "
            f"({summary.loc[2, 'Customer_Share']:.1f}%). "
            f"Mean annual spend: {summary.loc[2, 'Annual_Spend']:.2f}; "
            f"satisfaction: {summary.loc[2, 'Satisfaction']:.2f}/10; "
            f"basket value: {summary.loc[2, 'Basket_Value']:.2f}. "
            f"{membership_share(2, ['Gold', 'Platinum']):.1f}% are "
            "Gold or Platinum members."
        ),
        "Business_Decision": (
            "Retain high-value customers and protect their long-term "
            "contribution to the business."
        ),
        "Priority_Actions": (
            "Test personalized loyalty benefits and premium service; "
            "recommend complementary products; maintain a consistent "
            "online and in-store experience."
        ),
        "KPI_Baseline": (
            f"Spend: {summary.loc[2, 'Annual_Spend']:.2f}; "
            f"basket value: {summary.loc[2, 'Basket_Value']:.2f}; "
            f"satisfaction: {summary.loc[2, 'Satisfaction']:.2f}/10; "
            f"complaints: {summary.loc[2, 'Complaints']:.2f}."
        ),
        "Success_Measures": (
            "Retention rate, customer lifetime value, repeat purchase rate "
            "and contribution margin per customer."
        ),
        "Risk_and_Data_Need": (
            "Premium benefits may add cost without improving retention. "
            "Track customer outcomes across multiple periods and collect "
            "revenue and cost data."
        )
    },
    {
        "Segment": "3 — Established, Store-Oriented",
        "Evidence": (
            f"{summary.loc[3, 'Customers']:,} customers "
            f"({summary.loc[3, 'Customer_Share']:.1f}%). "
            f"Mean store visits: {summary.loc[3, 'Store_Visits']:.2f}/year; "
            f"online purchase share: {summary.loc[3, 'Online_Share']:.1f}%; "
            f"spend: {summary.loc[3, 'Annual_Spend']:.2f}; "
            f"tenure: {summary.loc[3, 'Tenure']:.2f} months. "
            f"{membership_share(3, ['Silver', 'Gold']):.1f}% are "
            "Silver or Gold members."
        ),
        "Business_Decision": (
            "Strengthen store engagement and increase customer value "
            "through relevant, convenient experiences."
        ),
        "Priority_Actions": (
            "Test in-store loyalty benefits and complementary-product offers; "
            "make digital ordering and account features convenient without "
            "forcing a shift away from physical stores."
        ),
        "KPI_Baseline": (
            f"Store visits: {summary.loc[3, 'Store_Visits']:.2f}/year; "
            f"spend: {summary.loc[3, 'Annual_Spend']:.2f}; "
            f"basket value: {summary.loc[3, 'Basket_Value']:.2f}; "
            f"satisfaction: {summary.loc[3, 'Satisfaction']:.2f}/10."
        ),
        "Success_Measures": (
            "Repeat store visits, average basket value, retention rate "
            "and digital adoption rate."
        ),
        "Risk_and_Data_Need": (
            "Frequent store visits do not necessarily imply higher profitability. "
            "Track transaction-level purchases, margins and adoption of "
            "digital features."
        )
    }
]

business_decision_matrix = pd.DataFrame(decision_data)

# 4. Display the integrated matrix as one report
display(
    business_decision_matrix.style
    .set_properties(**{
        "white-space": "pre-wrap",
        "text-align": "left",
        "vertical-align": "top",
        "min-width": "170px",
        "max-width": "280px"
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "left"),
                ("white-space", "normal")
            ]
        }
    ])
)

,Segment,Evidence,Business_Decision,Priority_Actions,KPI_Baseline,Success_Measures,Risk_and_Data_Need
0,"1 — Digital, Promotion-Dependent","3,028 customers (58.2%). Mean annual spend: 214.92; satisfaction: 7.00/10; complaints: 2.31; online purchase share: 57.8%; promotion purchase share: 50.4%. 96.5% are Basic or Silver members.",Improve profitable repeat purchasing and address customer experience problems without relying on blanket discounts.,Test targeted offers against a control group; investigate complaint drivers; introduce relevant loyalty milestones and complementary-product recommendations.,Spend: 214.92; basket value: 2353.96; satisfaction: 7.00/10; complaints: 2.31.,"Repeat purchase rate, incremental contribution margin, promotion redemption, average basket value and complaint rate.","Discounts may reduce margins without creating incremental sales. Collect offer exposure, redemption, transaction and cost data."
1,"2 — High-Value, Omnichannel","1,076 customers (20.7%). Mean annual spend: 523.65; satisfaction: 8.34/10; basket value: 6155.74. 79.6% are Gold or Platinum members.",Retain high-value customers and protect their long-term contribution to the business.,Test personalized loyalty benefits and premium service; recommend complementary products; maintain a consistent online and in-store experience.,Spend: 523.65; basket value: 6155.74; satisfaction: 8.34/10; complaints: 1.04.,"Retention rate, customer lifetime value, repeat purchase rate and contribution margin per customer.",Premium benefits may add cost without improving retention. Track customer outcomes across multiple periods and collect revenue and cost data.
2,"3 — Established, Store-Oriented","1,096 customers (21.1%). Mean store visits: 35.02/year; online purchase share: 19.6%; spend: 329.47; tenure: 111.55 months. 87.4% are Silver or Gold members.","Strengthen store engagement and increase customer value through relevant, convenient experiences.",Test in-store loyalty benefits and complementary-product offers; make digital ordering and account features convenient without forcing a shift away from physical stores.,Store visits: 35.02/year; spend: 329.47; basket value: 3626.23; satisfaction: 7.98/10.,"Repeat store visits, average basket value, retention rate and digital adoption rate.","Frequent store visits do not necessarily imply higher profitability. Track transaction-level purchases, margins and adoption of digital features."


## Overall Business Decision

The analysis identifies three actionable customer segments with different commercial priorities.

**First, prioritize profitable engagement for Segment 1.** Its large customer base creates an opportunity to test targeted promotions, improve satisfaction and reduce complaint burden. Campaigns should be evaluated against a control group to determine whether they generate incremental spending and contribution margin rather than merely subsidizing existing purchases.

**Second, protect customer value in Segment 2.** This segment has the highest observed spending and satisfaction and a strong concentration of Gold and Platinum members. Retention, personalized benefits and margin-aware cross-selling are the main priorities.

**Third, deepen engagement with Segment 3.** These customers have the highest store-visit frequency and longest tenure. In-store loyalty initiatives and convenient optional digital services should be tested to determine whether they improve customer value.

Membership tier is the strongest categorical differentiator of the three segments, whereas region, city tier, preferred category and acquisition channel provide little additional differentiation in this dataset. These attributes should not be the primary basis for separate segment strategies without further evidence.

### Measurement and Implementation

The existing dataset provides baseline indicators for spending, basket value, satisfaction, complaints and channel behavior. Repeat purchase rate, retention, offer redemption and incremental profitability require additional transaction, campaign or longitudinal data.

The next step is to pilot the recommended interventions, establish appropriate performance targets, and evaluate results using comparable control groups where feasible. The final allocation of marketing resources should be based on demonstrated incremental value, not on segment size or descriptive averages alone.